# Homework 4

Note \
This homework uses the pinned 2026 lead-scoring release in the course repository. The plan and report are available in cohorts/2026/data/. \
\
In this homework, we will use the 2026 lead-scoring dataset. Download it from here. \
https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv \
\
In this dataset our desired target for classification task will be <u>**converted**</u> variable - has the client signed up to the platform or not.
Data preparation\
\
    Check if the missing values are presented in the features.
    If there are missing values:
        For categorical features, replace them with 'NA'
        For numerical features, replace with with 0.0
\
Split the data into 3 parts with these exact calls: \

    df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1) \
    df_train, df_val = train_test_split( \
        df_full_train, test_size=0.25, random_state=1 \
    ) 
 \
## Question 1: ROC AUC feature importance
ROC AUC could also be used to evaluate feature importance of numerical variables. \
\
Let's do that\

    For each numerical variable, use it as score (aka prediction) and compute the AUC with the y variable as ground truth.
    Use the training dataset for that

If your AUC is < 0.5, invert this variable by putting "-" in front \
\
(e.g. -df_train['balance']) \
 \
AUC can go below 0.5 if the variable is negatively correlated with the target variable. You can change the direction of the correlation by negating this variable - then negative correlation becomes positive. \
 \
Which numerical variable (among the following 4) has the highest AUC? \
 
    lead_score
    number_of_courses_viewed
    interaction_count
    annual_income
\
## Question 2: Training the model

Apply one-hot-encoding using DictVectorizer and train the logistic regression with these parameters: \
\
LogisticRegression(solver='liblinear', C=1.0, max_iter=1000) \
\
What's the AUC of this model on the validation dataset? (round to 3 digits) \

    0.532
    0.632
    0.732
    0.832
\
## Question 3: Precision and Recall

Now let's compute precision and recall for our model. \
 
    Evaluate the model on all thresholds from 0.0 to 1.0 with step 0.01
    For each threshold, compute precision and recall
    Plot them
\
At which threshold precision and recall curves intersect? Ignore thresholds at which both metrics are zero, and choose the threshold with the smallest absolute difference between precision and recall. \

    0.43
    0.63
    0.73
    0.83
\
## Question 4: F1 score

Precision and recall are conflicting - when one grows, the other goes down. That's why they are often combined into the F1 score - a metrics that takes into account both \
\
This is the formula for computing F1:

    F 1 = 2 ⋅ P ⋅ R P + R
\
Where P is precision and R is recall. \
\
Let's compute F1 for all thresholds from 0.0 to 1.0 with increment 0.01 \
\
At which threshold F1 is maximal?

    0.21
    0.41
    0.61
    0.81
\
## Question 5: 5-Fold CV

Use the KFold class from Scikit-Learn to evaluate our model on 5 different folds: \

    KFold(n_splits=5, shuffle=True, random_state=1)

    Iterate over different folds of df_full_train
    Split the data into train and validation
    Train the model on train with these parameters: LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
    Use AUC to evaluate the model on validation
\
How large is standard deviation of the scores across different folds? \

    0.001
    0.007
    0.013
    0.060
\
## Question 6: Hyperparameter Tuning

Now let's use 5-Fold cross-validation to find the best parameter C \

    Iterate over the following C values: [0.000001, 0.001, 1]
    Initialize KFold with the same parameters as previously
    Use these parameters for the model: LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    Compute the mean ROC AUC as well as the std (round the mean and std to 3 decimal digits).
\
Which C leads to the best mean ROC AUC? \

    0.000001
    0.001
    1
\
If you have ties, select the score with the lowest std. If you still have ties, select the smallest C.
Learning in Public \
\
Share what you learned from this homework in a short post, screenshot, or notebook excerpt. Use #mlzoomcamp, tag Alexey Grigorev or DataTalksClub, and submit the post link with your homework. Don't share credentials, private data, or other secrets. \
\
See the learning in public guide for more ideas and examples.
Example post for LinkedIn

🚀 Completed Module 4 of Machine Learning Zoomcamp 2026!

I evaluated a classifier with ROC AUC, precision, recall, and F1, then used
cross-validation to measure score variation and tune a parameter.

My repo: <LINK>

#mlzoomcamp @Alexey Grigorev

Example post for X

Evaluated a classifier in Module 4 of #mlzoomcamp.

✅ Compared ROC AUC and threshold-based metrics
✅ Found a useful precision/recall trade-off
✅ Used 5-fold cross-validation for tuning

My repo: <LINK>

## Submit the results

    Submit your results here: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw04
    The numerical options are calculated from the pinned 2026 release. Use the value that matches your calculation; do not choose a merely close value.


# SETUP

In [38]:
##################################################################
# 0.1 - IMPORT PACKAGES
##################################################################
import numpy                      as     np                 # PROVIDES MATH DATA STRUCTURES AND FUNCTIONS
import pandas                     as     pd                 # PROVIDES BASIC ANALYTIC DATA STRUCTURE
import matplotlib.pyplot          as     plt                # USED FOR LINE GRAPHS
from   sklearn.linear_model       import LogisticRegression # 
from   sklearn.model_selection    import train_test_split   # USED TO SPLIT THE DATASET
from   sklearn.feature_extraction import DictVectorizer     # FUNCTION USED TO APPLY ONE-HOT ENCODING
from   sklearn.metrics            import (
                                    accuracy_score,         # 
                                    confusion_matrix,       # 
                                    classification_report,  # 
                                    roc_auc_score           # ROC - AREA UNDER THE CURVE 
                                  )
from   collections                import Counter

#   First, we load the data and clean it: 
#   - column names are lowercased, 
#   - spaces are replaced with underscores, 
#   - totalcharges is converted to numbers with errors='coerce' and 
#   - the missing values are filled with 0, and 
#   - the target is turned into 0/1:

##################################################################
# 0.2 - PREPARE THE DATASET
##################################################################
#   0.2.1 GET CSV AND LOAD DATA INTO A 'pandas' DATAFRAME
url             = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df              = pd.read_csv(url)

#   0.2.3 CONVERT COLUMN NAMES TO LOWERCASE AND REPLACE WHITESPACES WITH UNDERSCORES
df.columns      = df.columns.str.lower().str.replace(' ', '_')

##################################################################
# 0.3 - EXPLORATORY DATA ANALYSIS
##################################################################
# print(f"COLUMNS BY DATATYPE\r\n{df.dtypes}")
# print()
# print(f"INITIAL COUNT OF MISSING VALUES BY COLUMN \r\n{df.isna().sum()[df.isna().sum() > 0]}")

##################################################################
# 0.4 - CREATE LISTS OF NUMERICAL FEATURES AND CATEGORICAL FEATURES
##################################################################
target            = "converted"

numerical_columns = (
    df.dtypes[
        df.dtypes.isin([np.dtype("int64"), np.dtype("float64")])
    ]
      .index
      .difference([target])  # SKIP THE TARGET COLUMN
      .tolist()
)

categorical_columns = (
    df.select_dtypes(include=["str"])
      .columns
      .difference([target])
      .tolist()
)

##################################################################
# 0.5 - FILL MISSING VALUES 
##################################################################
#   0.5.1 - FIX NUMERICAL DATA VALUES
for col in numerical_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0) # FILL MISSING NUMERIC VALUES WITH ZEROS

#   0.5.2 - FIX CATEGORICAL DATA VALUES
for c in categorical_columns:
    df[c]       = df[c].fillna("NA")                      # FILL MISSING VALUES WITH "NA"
    df[c]       = df[c].str.lower().str.replace(' ', '_') # REPLACE WHITESPACES WITH UNDERSCORES 


##################################################################
# 0.6 - SPLIT THE DATATSET
##################################################################
#  SPLIT THE DATASET
#    SPLIT REQUIRES TWO PASSES
#      GOAL: 60% for training, 20% for validation and 20% for test. 
#    SPLIT IS DONE IN TWO STEPS 
#      1. SPLIT OFF INTO A TEST DATASET AND A NOT-TEST (df_full_train) DATASET 
#      2. SPLIT THE NOT-TEST DATASET INTO A TRAINING DATASET AND A VALIDATION DATASET 
#    USE 'random_state=1' SO THE SHUFFLE IS REPRODUCIBLE
df_full_train, df_test = train_test_split(
    df, 
    test_size=0.2, 
    random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, 
    test_size=0.25, 
    random_state=1
)
##################################################################
# 0.7 - USE ONE-HOT ENCODING TO CREATE FEATURE AND TARGET DATASETS
##################################################################
# REMINDER: THIS IS THE FUNCTION USED FOR ONE-HOT ENCODING
# from sklearn.feature_extraction import DictVectorizer

#   0.7.1 - CREATE DISK VECTORIZER INSTANCE 
dv         = DictVectorizer(sparse=False) 

#   0.7.2 - TRAINING DATASETS
#     0.7.2.1 - CREATE TRAINING DICTIONARY
train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')

#     0.7.2.2 - CREATE ONE-HOT ENCODED TRAINING DICTIONARY COLUMNS
X_train    = dv.fit_transform(train_dict)

#     0.7.2.3 - CREATE TRAIN TARGET VECTOR
y_train    = df_train[target].values
y_train    = df_train[target].to_numpy()

#   0.7.3 - TRAINING DATASETS
#     0.7.3.1 - CREATE VALIDATION DICTIONARY
val_dict   = df_val[categorical_columns + numerical_columns].to_dict(orient='records')

#     0.7.3.2 - CREATE ONE-HOT ENCODED VALIDATION DICTIONARY COLUMNS
X_val      = dv.transform(val_dict)

#     0.7.3.3 - CREATE TRAIN TARGET VECTOR
y_val      = df_val[target].values

##################################################################
# 0.8 - CREATE THE MODEL
##################################################################
# REMINDER: from   sklearn.linear_model       import LogisticRegression
model = LogisticRegression(solver='lbfgs' , max_iter=10000)
# DEFAULT SOLVER IN NEWER VERSIONS OF 'sklearn': solver='lbfgs'
#   MUST SPECIFY SOLVER EXPLICITLY IN OLDER VERSIONS

# 'fit()' CREATES A MODEL BRIDGING 'X_train' FEATURES TO 'y_train' TARGETS
model.fit(X_train, y_train)


##################################################################
# 0.9 - MAKE PREDICTIONS
##################################################################
# MAKE PREDICTIONS FOR THE VALIDATION DATASET BASED UPON THE TRAINING DATASET
y_pred       = model.predict_proba(X_val)[:, 1]
y_pred_proba = model.predict_proba(X_val)[:, 1]
y_pred       = (y_pred_proba >= 0.5).astype(int)

# Question 1: ROC AUC feature importance

**ROC AUC** could also be used to evaluate feature importance of numerical variables. \
\
Let's do that

    For each numerical variable, use it as score (aka prediction) and compute the AUC with the y variable as ground truth.
    Use the training dataset for that
 
If your AUC is < 0.5, invert this variable by putting "-" in front \
 \
(e.g. -df_train['balance']) \
 \
AUC can go below 0.5 if the variable is negatively correlated with the target variable. You can change the direction of the correlation by negating this variable - then negative correlation becomes positive. \
 \
Which numerical variable (among the following 4) has the highest AUC?

    * lead_score
    * number_of_courses_viewed
    * interaction_count
    * annual_income


In [51]:
#Q1 WORK
# REMINDER: SOLUTION USES 'from sklearn.metrics import roc_auc_score'

auc_max        = -1.0
for col in numerical_columns:
    scores     = df_train[col].to_numpy()
    auc        = roc_auc_score(y_train, scores)
    
    if auc > auc_max: 
        auc_max = auc
        col_max = col
        
    if auc < 0.5:
        scores = -scores                          # NEGATE 'score' VALUES
        auc    = roc_auc_score(y_train, scores)   # RECALCULATE AUC USING NEGATIVE 'score' VALUES
        if auc > auc_max: 
            auc_max = auc
            col_max = col
            
    print(f"{col}: AUC \t= {auc:.3f}")
print()
print(f"ANSWER:\r\n\t'{col_max}' HAS THE GREATEST AUC ({auc_max})")
# ANSWER:
#    'lead_score' HAS THE GREATEST AUC VALUE

annual_income: AUC 	= 0.608
interaction_count: AUC 	= 0.765
lead_score: AUC 	= 0.788
number_of_courses_viewed: AUC 	= 0.723

ANSWER:
	'lead_score' HAS THE GREATEST AUC (0.7882254810906102)


In [39]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_val, y_pred)


0.6962086362961863

In [27]:
# DEBUG 
# print(f"COUNT OF MISSING VALUES BY COLUMN \r\n{df.isna().sum()[df.isna().sum() > 0]}")

COUNT OF MISSING VALUES BY COLUMN 
Series([], dtype: int64)


In [30]:
# DEBUG 
# MAKE PREDICTIONS FOR THE VALIDATION DATASET BASED UPON THE TRAINING DATASET
y_pred   = model.predict_proba(X_val)[:, 1]

In [19]:
# DEBUG 


In [ ]:
# DEBUG 


In [ ]:
# DEBUG 


In [4]:
# DEBUG 


In [35]:
# DEBUG 
target            = "converted"

numerical_columns = (
    df.dtypes[
        df.dtypes.isin([np.dtype("int64"), np.dtype("float64")])
    ]
      .index
      .difference([target])  # SKIP THE TARGET COLUMN
      .tolist()
)

categorical_columns = (
    df.select_dtypes(include=["str"])
      .columns
      .difference([target])
      .tolist()
)


In [36]:
numerical_columns

['annual_income',
 'interaction_count',
 'lead_score',
 'number_of_courses_viewed']

In [37]:
categorical_columns

['employment_status', 'industry', 'lead_source', 'location']